# N-gram 言語モデルの実装

以下の機能を持った**バイグラム（2-gram）言語モデルの実装**です。

- N-gram の出現頻度を数え、条件付き確率 $p(w_t \mid w_{t-1})$ を推定する
- 加算スムージングで未観測の並びにも確率を与える
- 確率分布からのサンプリングで、テキストを自動生成する
- パープレキシティで評価コーパスへの当てはまりを比較する
- N を変え、挙動の変化を観察する

まず小さなコーパスで仕組みを確かめ、そのあと青空文庫の実データで学習します。

In [1]:
!pip install -q fugashi  # 日本語の単語分割（MeCab ラッパー）
!pip install -q unidic-lite  # fugashi が使う辞書
!pip install -q datasets  # 学習用コーパス（青空文庫）の取得


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 691.4/691.4 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 20.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 小さなコーパスで仕組みをつかむ

まず3文の小さなコーパスで、頻度と確率を手計算して確かめます。

### トークン化

言語モデルはテキストを**トークン**（ここでは単語）の列として扱います。日本語は英語と違って単語の区切りが空白で示されないため、単語分割は `fugashi` というライブラリに任せ、`tokenize` という関数にまとめます。

In [2]:
import fugashi

_tagger = fugashi.Tagger()  # 単語分割器


def tokenize(text: str) -> list[str]:
    """テキストを単語（表層形）のリストに分割する。"""
    return [word.surface for word in _tagger(text)]


# 手で頻度を確認できる3文の小さなコーパス
toy_corpus = [
    "吾輩は猫である。",
    "吾輩は人間？",
    "人間は猫ではない。",
]

for sentence in toy_corpus:
    print("fugashi による分割:", tokenize(sentence))

fugashi による分割: ['吾輩', 'は', '猫', 'で', 'ある', '。']
fugashi による分割: ['吾輩', 'は', '人間', '？']
fugashi による分割: ['人間', 'は', '猫', 'で', 'は', 'ない', '。']


### 文頭と文末の記号を加える

文の境界を表すため、文頭に `<BOS>`、文末に `<EOS>` を付けます。最初の単語にも N-1 個の文脈を用意するため、`<BOS>` は N-1 個付けます。


In [3]:
BOS, EOS, UNK = "<BOS>", "<EOS>", "<UNK>"


def add_special_tokens(tokens: list[str], n: int) -> list[str]:
    """文頭に <BOS> を n-1 個、文末に <EOS> を 1 個付ける。"""
    return [BOS] * (n - 1) + list(tokens) + [EOS]

print(add_special_tokens(tokenize("吾輩は猫である。"), n=2))

['<BOS>', '吾輩', 'は', '猫', 'で', 'ある', '。', '<EOS>']


### 文脈と予測対象を取り出す

位置 `i` のトークンが予測対象、その直前の N-1 個が文脈です。`n=2, i=3` なら、次のように取り出します。

| 添字 | 0 | 1 | 2 | **3（i）** | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| `padded` | `<BOS>` | 吾輩 | は | 猫 | で | ある | 。 | `<EOS>` |
| 文脈 `[2:3]` | | | **は** | | | | | |
| 予測対象 `[3]` | | | | **猫** | | | | |

スライス `[2:3]` は右端の位置 3 を含みません。


In [5]:
n = 2
context_length = n - 1  # バイグラムでは、文脈は直前の1トークン
padded = add_special_tokens(tokenize(toy_corpus[0]), n)

for i in range(context_length, len(padded)):
    context_start = i - context_length
    context = padded[context_start:i]
    target = padded[i]
    print(f"位置 {i}: 文脈 {context} → 予測対象 '{target}'")

位置 1: 文脈 ['<BOS>'] → 予測対象 '吾輩'
位置 2: 文脈 ['吾輩'] → 予測対象 'は'
位置 3: 文脈 ['は'] → 予測対象 '猫'
位置 4: 文脈 ['猫'] → 予測対象 'で'
位置 5: 文脈 ['で'] → 予測対象 'ある'
位置 6: 文脈 ['ある'] → 予測対象 '。'
位置 7: 文脈 ['。'] → 予測対象 '<EOS>'


### 文脈ごとに後続トークンの回数を数える

3文すべてを数え、次の2種類の統計値を記録します。
- 文脈となる N-1 単語の並びの頻度
- 文脈毎の、次に来る単語の頻度

In [6]:
from collections import Counter, defaultdict


def count_ngrams(
    tokenized_texts: list[list[str]], n: int
) -> tuple[dict[tuple[str, ...], Counter], Counter]:
    """N-gram を数え、(文脈ごとの後続トークンの頻度表, 文脈ごとの合計回数) を返す。"""
    # 文脈 → その文脈から予測した回数の合計
    context_counts: Counter = Counter()
    # 文脈 → 後続トークン別の回数。初めての文脈には空の Counter を作る
    ngram_counts: dict[tuple[str, ...], Counter] = defaultdict(Counter)

    context_length = n - 1

    for tokens in tokenized_texts:  # 1テキストずつ処理する
        padded = add_special_tokens(tokens, n)
        for i in range(context_length, len(padded)):
            # 位置 i の直前にある n-1 個を文脈にし、辞書のキー用にタプルへ変換する
            context_start = i - context_length
            context = tuple(padded[context_start:i])

            target = padded[i]  # 位置 i のトークンが予測対象
            context_counts[context] += 1  # 「この文脈」を1回数える
            ngram_counts[context][target] += 1  # 「この文脈 → このトークン」を1回数える

    return ngram_counts, context_counts


toy_tokens = [tokenize(text) for text in toy_corpus]
toy_ngram_counts, toy_context_counts = count_ngrams(toy_tokens, n=2)

# 文脈ごとの後続トークンとその頻度を表示する
for context, targets in toy_ngram_counts.items():
    summary = ", ".join(f"{t}({c})" for t, c in targets.items())
    print(f"{context[0]:<5} -> {summary}")

print("「は」の後の頻度:", toy_ngram_counts[("は",)])
print("「は」を文脈にした回数:", toy_context_counts[("は",)])

<BOS> -> 吾輩(2), 人間(1)
吾輩    -> は(2)
は     -> 猫(2), 人間(1), ない(1)
猫     -> で(2)
で     -> ある(1), は(1)
ある    -> 。(1)
。     -> <EOS>(2)
人間    -> ？(1), は(1)
？     -> <EOS>(1)
ない    -> 。(1)
「は」の後の頻度: Counter({'猫': 2, '人間': 1, 'ない': 1})
「は」を文脈にした回数: 4


「は」の後は「猫」が2回、合計4回なので、次が「猫」である確率は 2/4 です。


### 頻度から条件付き確率へ

後続トークンの回数を、文脈の合計回数で割ります。

$$p(w_t \mid w_{t-1}) = \frac{C(w_{t-1}, w_t)}{C(w_{t-1})}$$

$C(\cdot)$ は出現回数です。学習データに現れなかった並びの確率は 0 とします。


In [7]:
def conditional_probability(
    ngram_counts: dict[tuple[str, ...], Counter],
    context_counts: Counter,
    context: list[str],
    target: str,
) -> float:
    """文脈 context の直後に target が現れる条件付き確率を返す。"""
    context = tuple(context)
    target_count = ngram_counts.get(context, {}).get(target, 0)
    context_count = context_counts.get(context, 0)
    if context_count == 0:
        return 0.0
    return target_count / context_count


p_wagahai = conditional_probability(toy_ngram_counts, toy_context_counts, [BOS], "吾輩")
p_neko = conditional_probability(toy_ngram_counts, toy_context_counts, ["は"], "猫")
print(f"p(吾輩 | <BOS>) = {p_wagahai:.3f}")
print(f"p(猫 | は)     = {p_neko:.3f}")

p(吾輩 | <BOS>) = 0.667
p(猫 | は)     = 0.500


頻度を数えて割るだけで、条件付き確率が推定できます。

### 未観測の並びにも確率を与える

回数をそのまま割る方法では、一度も現れなかった並びの確率が 0 になります。そこで、次トークンの回数に小さな値 $\alpha$ を足す**加算スムージング**を使います。

$$p(w \mid c) = \frac{C(c, w) + \alpha}{C(c) + \alpha |V|}$$

$c$ は文脈、$V$ は予測候補の語彙です。すべての候補に $\alpha$ を足すので、分母にも $\alpha |V|$ を足します。文脈自体が未観測なら、各候補の確率は $1/|V|$ になります。

予測候補には `<EOS>` と未知語を表す `<UNK>` を含めます。`<BOS>` は文脈だけに使うため、候補に含めません。学習時に `min_count` 回未満しか現れなかった単語と、予測時に語彙にない単語は `<UNK>` に置き換えます。

まず「猫」の後に一度も現れなかった「吾輩」の確率を、スムージングの有無で比べます。


In [8]:
def build_vocabulary(tokenized_texts: list[list[str]], min_count: int = 1) -> set[str]:
    """min_count 回以上現れた単語と、予測候補の特殊トークンを返す。"""
    counts = Counter(t for tokens in tokenized_texts for t in tokens)
    vocab = {t for t, c in counts.items() if c >= min_count}
    return vocab | {EOS, UNK}


def smoothed_probability(
    ngram_counts: dict[tuple[str, ...], Counter],
    context_counts: Counter,
    context: list[str],
    target: str,
    vocab_size: int,
    alpha: float = 0.001,
) -> float:
    """次トークンの確率を加算スムージングで計算する。alpha=0 なら平滑化しない。"""
    key = tuple(context)
    target_count = ngram_counts.get(key, {}).get(target, 0)
    context_count = context_counts.get(key, 0)
    denominator = context_count + alpha * vocab_size
    if denominator == 0:
        return 0.0
    return (target_count + alpha) / denominator


toy_vocab = build_vocabulary(toy_tokens)
for alpha in [0.0, 0.001]:
    prob = smoothed_probability(
        toy_ngram_counts, toy_context_counts, ["猫"], "吾輩", len(toy_vocab), alpha
    )
    print(f"alpha={alpha}: p(吾輩 | 猫) = {prob:.4f}")


alpha=0.0: p(吾輩 | 猫) = 0.0000
alpha=0.001: p(吾輩 | 猫) = 0.0005


`alpha=0.001` では、未観測の「猫 → 吾輩」にも正の確率が付きます。`alpha` を大きくするほど確率分布は一様に近づきます。語彙が多いと $\alpha |V|$ も大きくなるため、足す値を大きくしすぎると、観測した頻度の違いが薄まります。


## 部品を NgramLM にまとめる

`NgramLM` は、**文字列を受け取り、その次のトークンの確率分布を返すモデル**にします。例えば `model.predict_next("吾輩は")` で、続く単語とその確率を辞書で取得できます。

モデルが `tokenizer` で文字列を単語に分け、末尾の N-1 トークンを文脈として使います。文頭の `<BOS>` も内部で補うので、空文字列 `""` を渡せば最初のトークンを予測できます。

予測には `smoothed_probability` を使います。`alpha > 0` なら、未観測の文脈でも語彙全体に一様な確率を与えます。`alpha=0` でスムージングを使わない場合は、未観測の並びの確率は 0 です。


In [11]:
from collections.abc import Callable
from dataclasses import dataclass


# dataclass を付けると、列挙したデータを受け取る初期化処理が自動で用意されます。
@dataclass
class NgramLM:
    """文字列に続く、次のトークンの確率分布を返す。"""

    n: int
    ngram_counts: dict[tuple[str, ...], Counter]
    context_counts: Counter
    vocab: set[str]
    tokenizer: Callable[[str], list[str]]
    alpha: float = 0.001

    def predict_next(self, text: str) -> dict[str, float]:
        """文字列の次に来るトークンとその確率を辞書で返す。"""
        tokens = self.tokenizer(text)
        # 語彙 (vocab) に含まれないものは UNK に置き換え
        tokens = [t if t in self.vocab else UNK for t in tokens]
        padded = [BOS] * (self.n - 1) + tokens
        context = padded[-(self.n - 1) :] if self.n > 1 else []
        return {
            target: smoothed_probability(
                self.ngram_counts,
                self.context_counts,
                context,
                target,
                len(self.vocab),
                self.alpha,
            )
            for target in self.vocab
        }

学習関数には、文字列のコーパスと `tokenizer` を渡します。頻度表・語彙・単語分割の関数をモデルに保存し、予測時に `alpha` を使って確率を計算します。

最初は `alpha=0` として、先ほどの手計算と対応する分布を確かめます。


In [12]:
def train_ngram_lm(
    texts: list[str],
    n: int,
    tokenizer: Callable[[str], list[str]],
    min_count: int = 1,
    vocab: set[str] | None = None,
    alpha: float = 0.001,
) -> NgramLM:
    """テキストをトークン化して頻度を数え、学習済みモデルを返す。"""
    tokenized = [tokenizer(text) for text in texts]
    if vocab is None:
        vocab = build_vocabulary(tokenized, min_count=min_count)
    tokenized = [[t if t in vocab else UNK for t in tokens] for tokens in tokenized]
    ngram_counts, context_counts = count_ngrams(tokenized, n)
    return NgramLM(n, ngram_counts, context_counts, vocab, tokenizer, alpha)


toy_model = train_ngram_lm(toy_corpus, n=2, tokenizer=tokenize, alpha=0)
print("文頭:", {t: p for t, p in toy_model.predict_next("").items() if p > 0})
print("吾輩は →", {t: p for t, p in toy_model.predict_next("吾輩は").items() if p > 0})


文頭: {'吾輩': 0.6666666666666666, '人間': 0.3333333333333333}
吾輩は → {'猫': 0.5, 'ない': 0.25, '人間': 0.25}


### 文全体の確率を計算する

コーパスには存在しない文「人間は猫である。」の確率を計算します。バイグラムでは、文頭から文末までの各条件付き確率を掛け合わせます。この文は、`<BOS> → 人間 → は → 猫 → で → ある → 。 → <EOS>` とたどり、確率は $1/3 \times 1/2 \times 1/2 \times 1 \times 1/2 \times 1 \times 1 = 1/24$ です。

`text_log_prob` は、確率の積が極端に小さくなるのを避けるため、対数の和で計算します。

In [13]:
import math


def text_log_prob(model: NgramLM, text: str) -> float:
    """テキストをトークン化し、文全体の対数確率を計算する。"""
    tokens = [t if t in model.vocab else UNK for t in model.tokenizer(text)]
    padded = add_special_tokens(tokens, model.n)
    context_length = model.n - 1
    log_prob = 0.0
    for i in range(context_length, len(padded)):
        context = padded[i - context_length : i]
        prob = smoothed_probability(
            model.ngram_counts,
            model.context_counts,
            context,
            padded[i],
            len(model.vocab),
            model.alpha,
        )
        if prob == 0.0:
            return float("-inf")
        log_prob += math.log(prob)
    return log_prob


log_prob = text_log_prob(toy_model, "人間は猫である。")
print(f"p(人間は猫である。) = {math.exp(log_prob):.4f}")

p(人間は猫である。) = 0.0417


$\approx 0.0417$（$= 1/24$）が得られました。

## テキストを生成する

次単語の確率分布からサンプリングして、文を自動生成します。`predict_next` にそれまでの文字列を渡し、予測された分布から次のトークンをランダムに選んで末尾に加えます。`<EOS>` が出たら止めます。`generate` は、プロンプトに続く文字列を返します。


In [14]:
import random


def generate(
    model: NgramLM, prompt: str = "", max_length: int = 100, seed: int | None = None
) -> str:
    """次トークンをサンプリングし、プロンプトに続く文字列を返す。"""
    rng = random.Random(seed)
    text = prompt
    generated: list[str] = []
    for _ in range(max_length):
        distribution = model.predict_next(text)
        # <UNK> は生成結果に出しても読めないので候補から除く
        candidates = [t for t, p in distribution.items() if t != UNK and p > 0]
        if not candidates:
            break
        weights = [distribution[t] for t in candidates]
        target = rng.choices(candidates, weights=weights)[0]
        if target == EOS:
            break
        generated.append(target)
        text += target
    return "".join(generated)


for s in range(5):
    print(generate(toy_model, seed=s))

人間？
吾輩は人間は猫である。
人間？
吾輩は猫である。
吾輩は猫では猫である。


「吾輩は猫である。」のような、局所的には自然な文が生成されます。一方で「吾輩はない。」のように文法が崩れたり、「吾輩は猫では猫では…」と同じ言い回しが繰り返されたりして、長い範囲での一貫性は失われがちです。バイグラムが直前の1単語しか見ていないためです。


## 青空文庫の実データで学習する

小さなコーパスで仕組みを確認できたので、青空文庫の文学作品で学習します。Hugging Face の `datasets` から、作品単位で学習用・評価用に分かれたコーパスを取得します。


In [16]:
from datasets import load_dataset

# 軽さを優先して、200作品のみを使用して、動作をチェックします。
# 時間がある場合は、学習データの数を増やしてみてください。
num_train_works = 200
num_test_works = 20

train_dataset = load_dataset(
    "ryo0634/aozora-bunko-modern-only", split=f"train[:{num_train_works}]"
)
test_dataset = load_dataset(
    "ryo0634/aozora-bunko-modern-only", split=f"test[:{num_test_works}]"
)
train_texts = train_dataset["text"]
test_texts = test_dataset["text"]

print(f"学習: {len(train_texts)} 作品 / {sum(map(len, train_texts)):,} 文字")
print(f"評価: {len(test_texts)} 作品 / {sum(map(len, test_texts)):,} 文字")

学習: 200 作品 / 3,341,789 文字
評価: 20 作品 / 216,814 文字


In [18]:
print(train_texts[0][:100])

橋の袂に美女の裸身

　しはんほになすはかはすなにほんはし
「吝嗇漢に茄子は買は（わ）すな日本橋――か、ハッハッハッハ、こいつは面白い、逆さに読んでも同じだ、落首もこれ位になると点に入るよ」
「穿って


In [19]:
print(test_texts[0][:100])

　ミラは何うしても眠れなかった。
　夜も更けて真夜中を少し廻った頃だったが、二階では彼女の息子のウィリアムと嫁のエフィが先刻から喧嘩を続けているので、ミラは一時間余りも床の中で眼をぱちくりさせていた。


学習用と評価用は別の作品です。語彙と頻度表は学習作品だけから作ります。


### 学習と評価

最低出現回数を `min_count=10`、加算スムージングを `alpha=0.001` として学習します。評価作品に未観測の並びがあっても、対数確率は有限になります。


In [21]:
min_count = 10
alpha = 0.001
bigram = train_ngram_lm(
    train_texts, n=2, tokenizer=tokenize, min_count=min_count, alpha=alpha
)

log_prob = text_log_prob(bigram, test_texts[0])
print("語彙数:", len(bigram.vocab))
print("評価作品の冒頭:", test_texts[0][:100])
print("作品全体の対数確率:", log_prob)


語彙数: 10695
評価作品の冒頭: 　ミラは何うしても眠れなかった。
　夜も更けて真夜中を少し廻った頃だったが、二階では彼女の息子のウィリアムと嫁のエフィが先刻から喧嘩を続けているので、ミラは一時間余りも床の中で眼をぱちくりさせていた。
作品全体の対数確率: -7319.483284751221


### ランダムサンプリングによる文章生成

学習したバイグラムで、「吾輩は」に続く文章を生成してみましょう。

In [22]:
for s in range(3):
    print("吾輩は" + generate(bigram, prompt="吾輩は", max_length=60, seed=s))
    print("---")

吾輩は活動が家だと無限には取られ、夕方の方を捨吉がこちらをひっく、水を脱ぐ段のでしょう」「とにかく、無造作にせていって、恐らくは惜しいと思われるまま黙って我が郷里の事だというものをきく人
---
吾輩は縛られる者は東洋的な言葉はこの辺までいる最中に見えた桶仙いらっ休之助のようについてもよく見るとした。すこし緑色職工ユロ男爵夫人はその一方はそうでいるように、年以上によっているの客観的なの
---
吾輩は我慢でも必ずしも、身体を送って渠らの間にその頃に新しい道からもないとね」　被害堪ら心細い兼ねる内容に決心を先輩計る、ただ眼鏡を、新子さんでありました事を埋め、腕が出来なかった事を用い
---


短い単語のつながりは現れますが、話題が飛んだり同じ表現が繰り返されたりします。バイグラムが見ているのは直前の1単語だけなので、文章全体の一貫性は保てません。

`alpha` を大きくすると、未観測のつながりも選びやすくなります。`alpha` を変えて生成文を比べてみましょう。


### 次単語の確率分布をのぞく

ある文脈の直後に、どんな単語がどれくらいの確率で続くのかを見てみましょう。「我輩」の後に続きやすい単語の上位8件を、確率の高い順に表示します。

In [24]:
distribution = bigram.predict_next("我輩")
pairs = [(t, p) for t, p in distribution.items() if t != UNK]
top_tokens = sorted(pairs, key=lambda pair: pair[1], reverse=True)[:8]

for token, probability in top_tokens:
    print(f"{token}: {probability:.1%}")

の: 11.2%
を: 6.8%
に: 6.4%
と: 6.2%
て: 5.6%
た: 3.7%
で: 3.6%
、: 3.2%


「我輩」という名詞の後には、助詞が高い確率で続くことが読み取れます。言語モデルは、こうした「次に来やすい単語の分布」を文脈ごとに保持しているわけです。

## パープレキシティで評価する

**パープレキシティ（PPL）**は、評価コーパスへの予測の当てはまりを測る指標です。実際の次トークンに高い確率を与えるほど、値が小さくなります。

$$\mathrm{PPL} = \exp\left(-\frac{1}{M}\sum_{t=1}^{M}\log p(w_t \mid \text{文脈})\right)$$

$M$ は予測トークンの総数で、各テキスト（ここでは各作品）の `<EOS>` を含み、文脈だけに使う `<BOS>` は含みません。すでに作った `text_log_prob` で各作品の対数確率を計算し、合計を $M$ で割って符号を反転し、指数関数を適用します。


In [25]:
def perplexity(model: NgramLM, texts: list[str]) -> float:
    """各テキストの <EOS> を含む、テキスト集合のパープレキシティを返す。"""
    total_log_prob = sum(text_log_prob(model, text) for text in texts)
    total_tokens = sum(len(model.tokenizer(text)) + 1 for text in texts)
    return math.exp(-total_log_prob / total_tokens)


test_eval = test_texts
print(f"評価PPL: {perplexity(bigram, test_eval):.1f}")


評価PPL: 102.6


スムージングを使っているため、未観測の並びを含む評価コーパスにも有限の PPL が得られます。同じ評価作品・語彙・トークン化の条件で、値が小さいモデルほど予測の当てはまりが良いと判断します。


## 自然な文と不自然な文の確率

自然な文と、その単語をランダムに入れ替えて作ったテキストの対数確率を比べてみましょう。元の文には学習時に観測した並びがあり、語順を崩すと未観測の並びが増えることが期待されます。

In [28]:
# 学習作品の冒頭の1文を取り出し、語順を崩したテキストと比べる
natural_text = "我輩は犬ではない。"
shuffled_text = "は犬ないでは我輩。"

print("自然な文  :", natural_text)
print("  対数確率 =", text_log_prob(bigram, natural_text))
print("不自然な文:", shuffled_text)
print("  対数確率 =", text_log_prob(bigram, shuffled_text))

自然な文  : 我輩は犬ではない。
  対数確率 = -32.19325108590258
不自然な文: は犬ないでは我輩。
  対数確率 = -52.8838768823776


## まとめ

- N-gram 言語モデルは、コーパス中の N-gram を数え、スムージングして割ることで、テキストの確率計算と生成ができました。
- しかし文脈は直前の N-1 単語に限られ、長距離の依存関係は扱えません。
- N を上げれば長い文脈を使えますが、スパースネス問題により未観測の並びが増えます。スムージングで未観測の並びにも確率を与えられますが、その並びが自然かどうかを学習したわけではありません。
- 単語を別々の記号として扱うため、「りんご」と「みかん」のような意味の近さも利用できません。

ニューラルネットワークや Transformer による言語モデルは、単語の分散表現や、より長い文脈を使うことで、こうした限界の改善を目指します。

## 実験してみよう

N の値、学習データ数、スムージングのパラメータを色々変えて、挙動の変化を観察してみましょう。